In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from nuscenes import NuScenes

In [5]:
nuscene_mini_data = NuScenes("v1.0-mini", dataroot="../data/v1.0-mini/")

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.248 seconds.
Reverse indexing ...
Done reverse indexing in 0.0 seconds.


In [6]:
type(nuscene_mini_data)

nuscenes.nuscenes.NuScenes

## The Data

That printout is the devkit loading all 13 JSON tables into memory and reporting the row count of each — it's confirming your mini data is intact. Every number there is a count of records in one table. Walking through them, grouped by what they actually describe:

**The taxonomy tables (small, fixed vocabularies):**
- **23 category** — the object classes that *can* be annotated (car, pedestrian, traffic_cone, bicycle, etc.). This is the label vocabulary, later collapsed to the 10 detection classes for the benchmark.
- **8 attribute** — extra state flags on annotations, e.g. `vehicle.parked`, `pedestrian.moving`, `cycle.with_rider`. The mAAE metric scores these.
- **4 visibility** — bins for how much of an object is visible (roughly 0–40%, 40–60%, 60–80%, 80–100%), from the camera views.

**The sensor/calibration tables:**
- **12 sensor** — the physical sensor *types* on the car: 1 LiDAR + 5 radar + 6 cameras = 12. This is the rig definition.
- **120 calibrated_sensor** — one calibration (extrinsics + intrinsics) per sensor *per vehicle/log*. 12 sensors × 10 logs ≈ 120. This is the **sensor→ego** half of your transform chain.

**The pose table:**
- **31206 ego_pose** — the car's global pose (position + orientation) at every sensor timestamp. This is the **ego→global** half of the chain. Note it matches `sample_data` exactly (both 31206) — there's one ego_pose per sensor reading, because the car has moved by the time each sensor fires.

**The structural / scene tables:**
- **8 log** — driving sessions (a log = one continuous recording from one car on one day). Mini draws its 10 scenes from 8 logs.
- **10 scene** — the 10 ~20-second clips that *define* the mini split. This is the number you sanity-check against ("did I get mini?").
- **404 sample** — **keyframes**: the annotated snapshots at 2 Hz. ~40 per scene × 10 ≈ 404. **This is your effective dataset size** — the thing you iterate over. (And why mini can't train a model: 404 samples is nothing.)
- **31206 sample_data** — *every* sensor reading, keyframes **and** the intermediate sweeps between them. Far more than `sample` because between each 2 Hz keyframe, LiDAR/radar/cameras keep firing at their own higher rates. Each row points to one file on disk (a `.pcd.bin`, `.jpg`, etc.).

**The annotation table:**
- **18538 sample_annotation** — individual 3D bounding boxes: one row per object instance per keyframe. ~46 boxes per sample on average. These are your **ground-truth labels**, given in the global frame (which is why Milestone 1 has you transform them into the LiDAR frame).

**The map table:**
- **4 map** — the map masks/layers associated with the logs' locations.

Two relationships worth locking in because they *are* the data model:

- **911 instance** vs **18538 sample_annotation** — an *instance* is one unique physical object (this specific parked car), tracked across the scene; each annotation is that object at one timestamp. So 911 real objects generate 18538 boxes as they're re-annotated frame to frame. Instance is the identity; annotation is the per-frame observation. (This is exactly what tracking tasks exploit.)
- **sample_data = sample + sweeps** — keyframes are the annotated 2 Hz subset; sample_data is the full firehose. You train detection on the keyframes; the sweeps exist for things like multi-sweep LiDAR accumulation.

**"Reverse indexing"** at the end is the devkit precomputing shortcut pointers between tables — e.g. so `sample` can directly list its annotations and sensor readings without you scanning the whole table each time. It's what makes `nusc.get(...)` and the `nusc.field2token` lookups fast. The "0.0 seconds" is just it being trivially quick on mini.

Net read: every table loaded, counts are sensible and internally consistent (10 scenes, 404 keyframes, ego_pose matching sample_data) — your mini split is complete and healthy. You're clear to start the Milestone 1 walk-through.